<a href="https://colab.research.google.com/github/veronicalaganaro/Pipeline-som-tico-com-GATK4-Mutect2/blob/main/Somaticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 5. Executar o Mutect2 Somático restringindo ao cromossomo 9
!./gatk-4.2.2.0/gatk Mutect2 \
    -R chr9.fa \
    -I tumor_JAK2.bam \
    -I normal_JAK2.bam \
    -normal WP044 \
    --germline-resource af-only-gnomad-chr9.vcf.gz \
    -L 9 \
    -O tumor_JAK2.somatic.vcf.gz

# 6. Calcular a contaminação para estimar artefatos restringindo ao cromossomo 9
!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I tumor_JAK2.bam \
    -V af-only-gnomad-chr9.vcf.gz \
    -L 9 \
    -O tumor_JAK2.table

!./gatk-4.2.2.0/gatk GetPileupSummaries \
    -I normal_JAK2.bam \
    -V af-only-gnomad-chr9.vcf.gz \
    -L 9 \
    -O normal_JAK2.table

!./gatk-4.2.2.0/gatk CalculateContamination \
    -I tumor_JAK2.table \
    -matched normal_JAK2.table \
    -O tumor_JAK2.contamination.table

# 7. Filtrar as variantes somáticas chamadas pelo Mutect2
!./gatk-4.2.2.0/gatk FilterMutectCalls \
    -R chr9.fa \
    -V tumor_JAK2.somatic.vcf.gz \
    --contamination-table tumor_JAK2.contamination.table \
    -O tumor_JAK2.filtered.vcf.gz

# Exibir as variantes identificadas que passaram nos filtros básicos
!zcat tumor_JAK2.filtered.vcf.gz | grep -v '##' | head -n 30

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R chr9.fa -I tumor_JAK2.bam -I normal_JAK2.bam -normal WP044 --germline-resource af-only-gnomad-chr9.vcf.gz -L 9 -O tumor_JAK2.somatic.vcf.gz
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:14:04.740 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
02:14:04.998

In [9]:
!samtools view -h -b /Volumes/Seagate\ Expansion\ Drive/data-lpfap10/projects/proadi/exome/bam/WP043/WP043.bam 9:5021937-5126899 | samtools bam2fq -1 tumor_R1.fq -2 tumor_R2.fq -

[E::hts_open_format] Failed to open file "/Volumes/Seagate Expansion Drive/data-lpfap10/projects/proadi/exome/bam/WP043/WP043.bam" : No such file or directory
samtools view: failed to open "/Volumes/Seagate Expansion Drive/data-lpfap10/projects/proadi/exome/bam/WP043/WP043.bam" for reading: No such file or directory
Failed to read header for "-"


### Método por Código: Fazer Upload do Arquivo BAM e BAI do seu computador
Execute a célula abaixo para selecionar e enviar os arquivos `WP043.bam` e o respectivo índice `.bai`.

In [10]:
from google.colab import files

print("Selecione o arquivo WP043.bam (e o arquivo .bai se necessário):")
uploaded = files.upload()

Selecione o arquivo WP043.bam (e o arquivo .bai se necessário):


Saving normal_JAK2.bam to normal_JAK2 (1).bam


### Executar o Samtools com os arquivos carregados
Após o upload terminar, o arquivo estará disponível na pasta `/content/`. Execute o comando abaixo para realizar a extração das leituras da região de interesse:

In [12]:
!samtools view -h -b /content/somatico/tumor_JAK2.bam 9:5021937-5126899 | samtools bam2fq -1 tumor_R1.fq -2 tumor_R2.fq -

@HWI-ST1374:105:H0NF0ADXX:1:2214:14248:24146_1:N:0:TAAGGCG
AGACACTGCGCCCAGCCCATTTGCAACTTTATTGTTTTCTCTTACAGGCAAATGTTCTGAAAAAGAC
+
:=4<;7<@<956<;;+/==5:<<+8<==;;;.9:==<;=>?>=9==8)>>=)<><=@?8,=;=<?=?
@HWI-ST1374:105:H0NF0ADXX:1:2212:11911:46910_1:N:0:TAAGGCG
GGTCAGATAATCTGCCTCAGATTTCCCAAGGGAATGGTAAAGATACACCTGAAGAACTGGATCTATTTGCTTCAT
+
DF>>=@<<<=<???@???>@=<==???>>@??=><??=<>>@=<<???@?@>>@=>>???=<??<<==>?<9840
@HWI-ST1374:105:H0NF0ADXX:1:2106:4235:45985_1:N:0:TAAGGCG
TTTTAATACTTACCACAAGCTTTAGAAGCAGCAATACAGATTTCTTCTGCAACATACTCCCCAGATGGAAA
+
DD>=;=<<>?=<>?>>>>@@?==<@=>@@>@@>><<>>@=<==??>@@?@>?>><<>?????>@=<??==;
@HWI-ST1374:105:H0NF0ADXX:1:2215:6743:2910_2:N:0:TAAGGCG
GTATTAGCATAGCATAATTAAAACAATCCATGCATAAAAAGGAAAATGCTGTTTTTTAATACTTACCACAAGCTTTAGAAGCAGCAATACAGATTTCTTNN
+
EC<<=;@@<;;@@=;:<==<>==>=>=@7=;@A<===>>=>@==>==?@>@=><>>>====?@=<>@=><;79100..++-/-00..-,0/31/005659:
@HWI-ST1374:105:H0NF0ADXX:1:1103:19036:61762_2:N:0:TAAGGCG
CAACAGAGCAAGACTCAGTCTCAAAAAAAA
+
6C=5;=;>><<><<=><=;=;>;<<<<<<<


In [15]:
!samtools view -h -b /content/somatico/normal_JAK2.bam 9:5021937-5126899 | samtools bam2fq -1 normal_R1.fq -2 normal_R2.fq -

@HWI-ST1374:105:H0NF0ADXX:2:1216:4559:71246_1:N:0:CGTACTA
GATTACAGGTGTGAGACACTGCGCCCAGCCCATTTGTAACTTTATTGTTTTCTCTTACAGGCAAATGTTCT
+
ED?>>@?AA>A<A=@=>>?@@@<@???A@???=>>@==>?@>>==>@=>>>@@@@>=??A?A?>>=B=>@=
@HWI-ST1374:105:H0NF0ADXX:2:1111:14711:95541_1:N:0:CGTACTA
GGAATGGCCTGCCTTACGATGACAGAAATGGAGGGAACATCCACCTCTTCTATATATCAGAATGGTGATATT
+
BB>;;@@@>??>@?=;>;=<@=>>@=>=<@?=@@3=>@==A@??@@@@9??=9<=;<??>>,<@:=A><;;5
@HWI-ST1374:105:H0NF0ADXX:2:1211:14324:30815_1:N:0:CGTACTA
CACCTCTTCTATATATCAGAATGGTGATATTTCTGGAAATGCCAATTCTATGAAGCAAATAGATCCAGTT
+
ED<=>002?><<<<<=>>?=<994<@=:=:::@?-5==?=????===A0<:?;;<@>=,5<6>=?@?@==
@HWI-ST1374:105:H0NF0ADXX:1:1116:3453:9291_1:N:0:CGTACTA
GGTAAAGATACACCTGAAGAACTGGATCTATTTGCTTCATAGAATTGGCATTTCCAGAAATATCACCATTCTG
+
E;><<=@===?>?>>@=>@==?@@?=<A?==>>@A>=A?=>A===>A@A?===A@=A>?>===B?@A?>==;8
@HWI-ST1374:105:H0NF0ADXX:2:1115:7649:41896_1:N:0:CATACTA
ATTCTATGAAGCAAATAGATCCAGTTCTTCAGGTGTATCTTTACCATTCCCTTGGGAAATCTGA
+
BA;<;<7?=>@@=6>9;6<=?=>@>>@?>?>@?38<<==@>:;;=/<>9?@96<

In [19]:
!echo "=== Verificando tumor_JAK2.bam ==="
!samtools quickcheck -v /content/somatico/tumor_JAK2.bam && echo "BAM OK" || echo "BAM Corrompido/Vazio"
!samtools view -H /content/somatico/tumor_JAK2.bam | head -n 5

!echo "\n=== Verificando normal_JAK2.bam ==="
!samtools quickcheck -v /content/somatico/normal_JAK2.bam && echo "BAM OK" || echo "BAM Corrompido/Vazio"
!samtools view -H /content/somatico/normal_JAK2.bam | head -n 5

=== Verificando tumor_JAK2.bam ===
/content/somatico/tumor_JAK2.bam was not identified as sequence data.
/content/somatico/tumor_JAK2.bam
BAM Corrompido/Vazio
[main_samview] fail to read the header from "/content/somatico/tumor_JAK2.bam".
\n=== Verificando normal_JAK2.bam ===
BAM OK
@HD	VN:1.4	GO:none	SO:coordinate
@SQ	SN:1	LN:249250621
@SQ	SN:2	LN:243199373
@SQ	SN:3	LN:198022430
@SQ	SN:4	LN:191154276


In [24]:
!samtools index tumor_JAK2.bam

In [21]:
!git checkout -- /content/somatico/tumor_JAK2.bam /content/somatico/tumor_JAK2.bam.bai
!echo "=== Validando integridade após restauração ==="
!samtools quickcheck -v /content/somatico/tumor_JAK2.bam && echo "BAM restaurado com sucesso!" || echo "Erro ao restaurar BAM."

=== Validando integridade após restauração ===
BAM restaurado com sucesso!


In [26]:
!samtools index normal_JAK2.bam

In [28]:
!zgrep -w "\#" af-only-gnomad.raw.sites.chr.vcf.gz > header

gzip: af-only-gnomad.raw.sites.chr.vcf.gz: No such file or directory


In [29]:
# Extrair o cabeçalho usando o arquivo correto do cromossomo 9 presente na pasta somatico
!zgrep -w "#" /content/somatico/af-only-gnomad-chr9.vcf.gz > /content/somatico/header
!head -n 20 /content/somatico/header

##fileformat=VCFv4.2
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=PL,Number=G,Type=Integer,Description="Normalized, Phred-scaled likelihoods for genotypes as defined in the VCF specification">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=LowQual,Description="">
##FILTER=<ID=InbreedingCoeff,Description="">
##FILTER=<ID=PASS+,Description="">
##FILTER=<ID=PASS+,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##FILTER=<ID=PASS,Description="">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Allele count in genotypes, for each ALT allele, in the same order as listed">
##INFO=<ID=AF,Number=A,Type=Float,Description

In [31]:
# Extrair a região do gene JAK2 utilizando o recurso correto do gnomAD para o chr9
!zgrep -w "^9" /content/somatico/af-only-gnomad-chr9.vcf.gz | awk '$2>=5021937 && $2<=5126899' > /content/somatico/JAK2.region
# Exibir as primeiras linhas da região extraída
!head -n 10 /content/somatico/JAK2.region

9	5021940	.	A	G	1902.40	PASS	AC=2;AF=4.923E-5
9	5021993	.	A	C	506.46	PASS	AC=1;AF=2.462E-5
9	5022035	.	C	T	1311.40	PASS	AC=2;AF=4.923E-5
9	5022042	rs150159583	A	G	219.46	PASS	AC=1;AF=2.462E-5
9	5022085	.	A	C	572.40	PASS	AC=2;AF=4.923E-5
9	5022091	.	T	C	1962.04	PASS	AC=4;AF=9.846E-5
9	5022098	.	A	G	621.40	PASS	AC=2;AF=4.923E-5
9	5022124	rs138655335	C	A	3052.45	PASS	AC=9;AF=2.215E-4
9	5022125	.	C	T	243.46	PASS	AC=1;AF=2.462E-5
9	5022130	rs143227399	G	A	2152.04	PASS	AC=4;AF=9.846E-5


In [32]:
!cat header JAK2.region > af-only-gnomad-chr9.vcf

In [34]:
# Mover o arquivo para a pasta correta caso tenha sido criado fora, compactar e indexar
!mv af-only-gnomad-chr9.vcf /content/somatico/af-only-gnomad-chr9.vcf 2>/dev/null || true
!bgzip -f /content/somatico/af-only-gnomad-chr9.vcf
!tabix -f -p vcf /content/somatico/af-only-gnomad-chr9.vcf.gz

# Confirmar a criação do arquivo compactado e seu respectivo índice
!ls -lh /content/somatico/af-only-gnomad-chr9.vcf.gz*

/bin/bash: line 1: bgzip: command not found
/bin/bash: line 1: tabix: command not found
-rw-r--r-- 1 root root 138K Nov 25 02:10 /content/somatico/af-only-gnomad-chr9.vcf.gz
-rw-r--r-- 1 root root  184 Nov 25 02:10 /content/somatico/af-only-gnomad-chr9.vcf.gz.tbi


In [35]:
!tabix -p vcf af-only-gnomad-chr9.vcf.gz

/bin/bash: line 1: tabix: command not found


In [36]:
# Indexar o arquivo VCF utilizando o GATK IndexFeatureFile
!./gatk-4.2.2.0/gatk IndexFeatureFile \
    -I /content/somatico/af-only-gnomad-chr9.vcf.gz

# Confirmar a existência do índice gerado (.tbi)
!ls -lh /content/somatico/af-only-gnomad-chr9.vcf.gz*

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar IndexFeatureFile -I /content/somatico/af-only-gnomad-chr9.vcf.gz
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:42:12.651 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
02:42:12.957 INFO  IndexFeatureFile - ------------------------------------------------------------

In [38]:
!sh run.chr13-chr19.sh tumor_JAK2.bam normal_JAK2.bam WP043 WP044
!sh run.chr13-chr19.sh tumor_wp017.bam tumor_wp018.bam WP017 WP018
!sh run.chr13-chr19.sh tumor_wp190.bam tumor_wp191.bam WP190 WP191

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19.fa -I tumor_JAK2.bam -I normal_JAK2.bam -normal WP044 --germline-resource af-only-gnomad-chr13-chr19.vcf.gz -O WP043.somatic.vcf.gz -L hg19.interval_list
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:43:02.423 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar!/com/intel/gkl/native/libgkl_compress

In [39]:
# 1. Baixar o genoma de referência hg19 completo compactado
!wget -c http://hgdownload.cse.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz

# 2. Descompactar o arquivo no diretório /content/somatico/
!zcat hg19.fa.gz > /content/somatico/hg19.fa

# 3. Criar os índices necessários para o GATK e Samtools
!samtools faidx /content/somatico/hg19.fa
!./gatk-4.2.2.0/gatk CreateSequenceDictionary -R /content/somatico/hg19.fa

# 4. Confirmar a criação dos arquivos de referência
!ls -lh /content/somatico/hg19.*

--2026-11-25 02:45:46--  http://hgdownload.cse.ucsc.edu/goldenPath/hg19/bigZips/hg19.fa.gz
Resolving hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.cse.ucsc.edu (hgdownload.cse.ucsc.edu)|128.114.119.163|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 948731419 (905M) [application/x-gzip]
Saving to: ‘hg19.fa.gz’

hg19.fa.gz          100%[===================>] 904.78M  17.9MB/s    in 57s     

2026-11-25 02:46:44 (15.8 MB/s) - ‘hg19.fa.gz’ saved [948731419/948731419]

Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar CreateSequenceDictionary -R /content/somatico/hg19.fa
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seem

In [40]:

# variaveis fixas
gnomad="af-only-gnomad-chr13-chr19.vcf.gz"
interval="hg19.interval_list"
genome="hg19.fa"
pon="Mutect2-exome-panel.vcf"

# bam e ids
tumor=$1
id_tumor=$3

# rodando mutect2
./gatk-4.2.2.0/gatk Mutect2 \
	-R $genome \
	-I $tumor \
	--germline-resource $gnomad \
	--panel-of-normals $pon \
	-O $id_tumor.somatic.pon.vcf.gz \
	-L $interval

# getPileup tumor
./gatk-4.2.2.0/gatk GetPileupSummaries \
	-I $tumor \
	-V $gnomad \
	-L $interval \
	-O $id_tumor.pon.table

# CalculateContamination
./gatk-4.2.2.0/gatk CalculateContamination \
	-I $id_tumor.pon.table \
	-O $id_tumor.contamination.pon.table

# FilterMutectCalls
./gatk-4.2.2.0/gatk FilterMutectCalls \
	-R $genome \
	-V $id_tumor.somatic.pon.vcf.gz \
	--contamination-table $id_tumor.contamination.pon.table \
	-O $id_tumor.filtered.pon.vcf.gz

SyntaxError: invalid syntax (1508782459.py, line 8)

In [41]:
# Mudar para o diretório de trabalho correto e executar o pipeline em lote para as três duplas de amostras
%cd /content/somatico/

print("=== Iniciando lote 1/3 (JAK2) ===")
!sh run.chr13-chr19.pon.sh tumor_JAK2.bam normal_JAK2.bam WP043 WP044

print("\n=== Iniciando lote 2/3 (WP017/WP018) ===")
!sh run.chr13-chr19.pon.sh tumor_wp017.bam tumor_wp018.bam WP017 WP018

print("\n=== Iniciando lote 3/3 (WP190/WP191) ===")
!sh run.chr13-chr19.pon.sh tumor_wp190.bam tumor_wp191.bam WP190 WP191

/content/somatico
=== Iniciando lote 1/3 (JAK2) ===
Using GATK jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/somatico/gatk-4.2.2.0/gatk-package-4.2.2.0-local.jar Mutect2 -R hg19.fa -I tumor_JAK2.bam --germline-resource af-only-gnomad-chr13-chr19.vcf.gz --panel-of-normals Mutect2-exome-panel.vcf -O WP043.somatic.pon.vcf.gz -L hg19.interval_list
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
02:48:57.978 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:/content/somatico/gatk-4.2.2.0/ga